In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In this Jupyter Notebook we compare performances of standard the variable inclusion method used for variable selection with a modified log-likelihood method.

### Standard Variable Inclusion
The BART algorithm produces a large number of tree ensambles (posterior samples), each containing _m_ trees. For each posterior sample we calculate the proportion of times each variable was used in a splitting rule among all splitting variables. Averaging the proportions across all posterior samples we estimate the _variable inclusion proportion_ of each variable.

An example for ensamble with $m=3$ trees can be seen in the figure below. Once we calculate inclusion proportions $v_k^{(s)}$ for all $S$ posterior samples, we get the estimated _variable inclusion proportions_ by averaging $p_k = \frac{1}{S}\sum_s^Sv_k^{(s)}$ for all $K$ variables. 

<img
  src="figures/bart_vi_standard.svg"
  alt="Variable inclusion proportions in BART"
  style="
    display: block;
    width: min(900px, 100%);
    height: auto;
    margin: 1rem auto;
  ">

Intuitively, a large variable inclusion proportion suggests importance of a given variable. This is especially clear when the number of trees in each sample (_m_) is small, forcing the algorithm to be selective with variables.

### Log-likelihood Variable Inclusion
The modified variable inclusion proportion calculates the estimated inclusion proportions in the same way as before, but each appearance of a variable in a splitting rule is weighted with a log-likelihood difference. The idea is that original estimated variable inclusions don't take into account how much the variables improve the fit of the algorithm, while the weights do.

For each internal node a log-likelihood difference is calculated between the current tree and a tree where the node in question is a leaf. Thus the weights represent how much the log-likelihood of a tree is imporved compared to if the split in question (and all subsequent splits) didn't happen. 

<img
  src="figures/bart_vi_weighted.svg"
  alt="Variable inclusion proportions in BART"
  style="
    display: block;
    height: 600px;
    max-width: 100%;
    margin: 1rem auto;
  ">


In [11]:
columns = [
    "scenario", "n", "p", "s2",
    "precision_raw_mean", "precision_logl_mean", "precision_diff",
    "recall_raw_mean", "recall_logl_mean", "recall_diff",
    "f1_raw_mean", "f1_logl_mean", "f1_diff",
]

diff_columns = [
    "precision_diff", "recall_diff", "f1_diff",
]

mean_se_columns = {
    "precision_raw_mean": "precision_raw_se",
    "precision_logl_mean": "precision_logl_se",
    "recall_raw_mean": "recall_raw_se",
    "recall_logl_mean": "recall_logl_se",
    "f1_raw_mean": "f1_raw_se",
    "f1_logl_mean": "f1_logl_se",
}


def show_results(results: pd.DataFrame):
    df = results.copy()

    df["precision_diff"] = (
        df["precision_logl_mean"] - df["precision_raw_mean"]
    )
    df["recall_diff"] = (
        df["recall_logl_mean"] - df["recall_raw_mean"]
    )
    df["f1_diff"] = (
        df["f1_logl_mean"] - df["f1_raw_mean"]
    )

    # Combine each mean with its standard error.
    for mean_col, se_col in mean_se_columns.items():
        df[mean_col] = [
            f"{mean:.3f} ({se:.2f})"
            for mean, se in zip(df[mean_col], df[se_col])
        ]

    df = df[columns]

    def highlight_diff(value):
        if value > 0:
            return "background-color: #6aa876; color: #1f1f1f;"
        if value < 0:
            return "background-color: #c74040; color: #1f1f1f;"
        return ""

    display(
        df.style
        .format(
            {
                "s2": "{:.1f}",
                "precision_diff": "{:.3f}",
                "recall_diff": "{:.3f}",
                "f1_diff": "{:.3f}",
            }
        )
        .map(highlight_diff, subset=diff_columns)
    )

In all simulations the standard additive model is assumed, ie. for a given input $x=(x_1,\dots, x_p)$ the response $Y$ is assumed to be:
$$
    Y = f(x) + \epsilon, \quad \epsilon \sim N(0, \sigma^2)
$$

Bellow are results of standard and weighted variable inclusion proportions for some simulated sets of inputs $x$ and functions $f$. For each set of inputs and outputs, a number of hyperparameter values were tested. Hyperparameters include size of sample (_n_), number of covariates (_p_) and variance of noise ($\sigma^2$). For detection thresholds (how high an inclusion proportion has to be in order to say a variable is "important") the method proposed in _Variable selection for BART: An application to gene regulation_ is used.

The reported scores (_precision_, _recall_, _f1_) are averaged over 100 repeats for each combination of hyperparameters.

The values in difference columns are marked green or red, depending on if the weighted or standard proportions performed better.

## Continuous Predictors and Continuous Response

In both of these examples $n = 500$ was considered, with $p \in \{50, 200\}$ and $\sigma^2 = 1$.

In the first scenario the Friedman function was used, defined as:
$$
    f(x) = 10 \sin{(\pi x_1 x_2)} + 20(x_3 - 0.5)^2 + 10x_4 + 5x_5,
$$
with $x_1, \dots, x_p \sim Unif(0, 1)$ independent.

In [12]:
cc1_results = pd.read_csv("results/cc1_001.csv")
show_results(cc1_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,cc1,500,50,1.0,0.954 (0.01),1.000 (0.00),0.046,0.926 (0.01),0.828 (0.01),-0.098,0.934 (0.01),0.901 (0.01),-0.033
1,cc1,500,200,1.0,0.452 (0.02),0.643 (0.01),0.191,0.750 (0.02),0.946 (0.01),0.196,0.550 (0.01),0.759 (0.01),0.208


For the second continuous-continuous scenario, detection for correlated covariates was compared.

Here $x_1,\dots,x_p$ came from a multivariate normal distribution $N(0, \Sigma)$ where $\Sigma_ij = 0.3^{|i - j|}$ and $f$ is defined as:
$$
    f(x) = 2 x_1 x_4 + 2 x_7 x_{10}

$$

In [13]:
cc2_results = pd.read_csv("results/cc2_001.csv")
show_results(cc2_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,cc2,500,50,1.0,0.204 (0.03),0.385 (0.03),0.181,0.135 (0.02),0.278 (0.02),0.143,0.156 (0.02),0.309 (0.02),0.153
1,cc2,500,200,1.0,0.047 (0.01),0.065 (0.01),0.018,0.077 (0.01),0.207 (0.02),0.130,0.057 (0.01),0.098 (0.01),0.041


## Mixed Predictors and Continuous Response

In these simulations some covariates come from a Bernoulli discrete distribution.

In the first scenario, predictors $x_1, \dots, x_{\lceil p/2 \rceil} \sim Bernoulli(0.5)$ independently and $x_{\lceil p/2 \rceil + 1}, \dots, x_p \sim Unif(0, 1)$ independently. Function $f$ is:
$$
    f(x) = 10 \sin{(\pi x_{\lceil p/2 \rceil +1} x_{\lceil p/2 \rceil +2})} + 20(x_{\lceil p/2 \rceil +3} - 0.5)^2 + 10x_1 + 5x_2
$$

Here we consider $n \in \{ 500, 1000 \}$, $p \in \{ 50, 200 \}$ and $\sigma^2 \in \{1, 10\}$.

In this scenario, predictors $x_1, \dots, x_{20} \sim Bernoulli(0.5)$, $x_{21}, \dots, x_{40} \sim Bernoulli(0.5)$ and $x_41, \dots, x_{84} \sim Bernoulli(0.5)$ independently, while $f$ is:
$$
    f(x) = -4 + x_1 + \sin(\pi x_1 x_{44}) - x_{21} + 0.6 x_{41} x_{42} - \exp[-2(x_{42} + 1)^2] - x_{43}^2 + 0.5x_{44}
$$

Here we consider $n \in \{500, 1000\}$ and $\sigma^2 \in \{1, 10\}$.

In [15]:
cm1_results = pd.read_csv("results/cm1_001.csv")
show_results(cm1_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,cm1,500,50,1.0,0.905 (0.01),0.998 (0.00),0.093,0.604 (0.01),0.800 (0.01),0.196,0.717 (0.01),0.884 (0.01),0.167
1,cm1,500,50,10.0,0.890 (0.02),0.998 (0.00),0.108,0.558 (0.01),0.818 (0.01),0.260,0.675 (0.01),0.896 (0.01),0.221
2,cm1,500,200,1.0,0.343 (0.01),0.641 (0.02),0.299,0.544 (0.01),0.906 (0.01),0.362,0.415 (0.01),0.741 (0.01),0.326
3,cm1,500,200,10.0,0.320 (0.01),0.547 (0.01),0.227,0.502 (0.01),0.910 (0.01),0.408,0.384 (0.01),0.675 (0.01),0.291
4,cm1,1000,50,1.0,0.946 (0.01),0.996 (0.00),0.051,0.634 (0.01),0.844 (0.01),0.210,0.754 (0.01),0.910 (0.01),0.156
5,cm1,1000,50,10.0,0.946 (0.01),0.998 (0.00),0.052,0.616 (0.01),0.820 (0.01),0.204,0.742 (0.01),0.897 (0.01),0.154
6,cm1,1000,200,1.0,0.393 (0.01),0.740 (0.02),0.347,0.614 (0.01),0.952 (0.01),0.338,0.470 (0.01),0.823 (0.01),0.353
7,cm1,1000,200,10.0,0.346 (0.01),0.689 (0.01),0.343,0.564 (0.01),0.936 (0.01),0.372,0.421 (0.01),0.786 (0.01),0.365


In [16]:
cm2_results = pd.read_csv("results/cm2_001.csv")
show_results(cm2_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,cm2,500,50,1.0,0.667 (0.02),0.938 (0.01),0.270,0.452 (0.01),0.675 (0.01),0.223,0.527 (0.01),0.777 (0.01),0.250
1,cm2,500,50,10.0,0.501 (0.03),0.646 (0.02),0.146,0.283 (0.01),0.495 (0.01),0.212,0.344 (0.01),0.548 (0.01),0.204
2,cm2,1000,50,1.0,0.757 (0.02),0.986 (0.01),0.229,0.502 (0.01),0.750 (0.01),0.248,0.593 (0.01),0.848 (0.01),0.255
3,cm2,1000,50,10.0,0.629 (0.02),0.867 (0.02),0.238,0.362 (0.01),0.610 (0.01),0.248,0.447 (0.01),0.704 (0.01),0.257


### Mixed Predictors and Binary Response

In the first scenario with a binary response, we sample predictors $x_1, \dots, x_{\lceil p/2 \rceil} \sim Bernoulli(0.5)$ independently and $x_{\lceil p/2 \rceil + 1}, \dots, x_p \sim Unif(0, 1)$ independently. We sample the reponse $y$ from $Bernoulli(\Phi(f(x)))$ where $f$ is the same as in the first mixed-continuous scenario.

We try combinations of $n \in \{500, 1000\}$ and $p \in \{50, 200\}$.

In [17]:
bm1_results = pd.read_csv("results/bm1_001.csv")
show_results(bm1_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,bm1,500,50,1.0,0.087 (0.02),0.152 (0.03),0.064,0.030 (0.01),0.070 (0.01),0.040,0.043 (0.01),0.092 (0.02),0.050
1,bm1,500,200,1.0,0.026 (0.01),0.037 (0.01),0.011,0.034 (0.01),0.068 (0.01),0.034,0.029 (0.01),0.047 (0.01),0.018
2,bm1,1000,50,1.0,0.072 (0.02),0.313 (0.04),0.241,0.030 (0.01),0.144 (0.02),0.114,0.041 (0.01),0.188 (0.02),0.147
3,bm1,1000,200,1.0,0.018 (0.01),0.050 (0.01),0.032,0.024 (0.01),0.100 (0.01),0.076,0.021 (0.01),0.066 (0.01),0.045


In this scenario, we sample predictors $x_1, \dots, x_{20} \sim Bernoulli(0.5)$, $x_{21}, \dots, x_{40} \sim Bernoulli(0.5)$ and $x_{41}, \dots, x_{84} \sim Bernoulli(0.5)$ independently and response $y$ from  $Bernoulli(\Phi(f(x)))$, where $f$ is the same as in the second mixed-continuous scenario.

Here we test $n \in \{500, 1000\}$.

In [18]:
bm2_results = pd.read_csv("results/bm2_001.csv")
show_results(bm2_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,bm2,500,50,1.0,0.075 (0.02),0.100 (0.02),0.025,0.033 (0.01),0.053 (0.01),0.020,0.045 (0.01),0.067 (0.01),0.022
1,bm2,1000,50,1.0,0.053 (0.01),0.122 (0.02),0.069,0.028 (0.01),0.070 (0.01),0.042,0.036 (0.01),0.084 (0.01),0.048
